# Experimento 6 — Remoção das Variáveis Categóricas com Balanceamento

O sexto experimento mantém exatamente o mesmo conjunto de variáveis de
entrada do Experimento 5 — apenas variáveis físico-químicas numéricas:

- `Temperature (cel)`
- `Orthophosphate (mg/l)`
- `Nitrogen (mg/l)`

A única diferença em relação ao Experimento 5 é a introdução do
parâmetro `class_weight='balanced'` no `LGBMClassifier`. Essa
configuração ajusta automaticamente os pesos das classes de forma
inversamente proporcional às suas frequências no conjunto de treino,
penalizando mais os erros cometidos nas classes minoritárias (`Crítica`
e `Atenção`) sem necessidade de reamostrar o dataset.

Nos experimentos anteriores sem balanceamento, o modelo tendia a
concentrar seu poder preditivo na classe majoritária (`Excelente`),
obtendo alta acurácia global às custas de recall quase nulo para as
classes raras. O Experimento 6 visa investigar se o ajuste de pesos
internos ao algoritmo é suficiente para mitigar esse comportamento
sem alterar a distribuição do dataset.

O objetivo deste experimento é investigar:

- se `class_weight='balanced'` melhora o recall das classes
  minoritárias (`Crítica` e `Atenção`) em relação ao Experimento 5;
- qual o custo desse ajuste sobre a acurácia global e o desempenho
  da classe `Excelente`;
- como o nível de overfitting se comporta com o balanceamento
  interno em comparação ao modelo sem balanceamento.

O dataset foi dividido em:

- 80% para treino
- 20% para teste

A separação foi realizada utilizando `train_test_split` com
`random_state=42` e `stratify=y`, mantendo os mesmos parâmetros
de todos os experimentos anteriores.

Após o treinamento, o modelo foi avaliado utilizando:

- Accuracy
- Precision
- Recall
- F1-score
- Matriz de confusão

## Preparação do ambiente

In [1]:
# IMPORT DE BIBLIOTECAS
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import os
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

SEED = 42

In [2]:
# DETECÇÃO DE AMBIENTE
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Ambiente Google Colab detectado.")
    drive.mount('/content/drive')
    DATA_PATH = Path(
        "/content/drive/MyDrive/EDA_AquaSense/Dataset/processed/amostra_rotulada.parquet"
    )
else:
    print("Ambiente local/VS Code detectado.")
    DATA_PATH = Path("../../dataset/amostra_rotulada.parquet")

df = pd.read_parquet(DATA_PATH)

print("Dataset Parquet carregado com sucesso.")
print(f"Shape do dataset: {df.shape}")

df.head()

Ambiente Google Colab detectado.
Mounted at /content/drive
Dataset Parquet carregado com sucesso.
Shape do dataset: (141399, 22)


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),...,CCME_Values,CCME_WQI,ph_ok,od_ok,dbo_ok,nitrate_ok,ammonia_limit,ammonia_ok,environmental_score,conama_status
0,Canada,CZPOH_1117,River,05-03-2012,0.092790,2.25455,9.43636,0.06100,7.50000,9.40000,...,93.116725,Good,1,1,1,1,3.7,1,5,Excelente
1,Canada,FISW_32,Lake,02-12-2003,0.043792,2.13333,9.82400,0.00200,7.79000,12.00000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
2,Canada,CZPOH_1036,River,12-03-2018,0.085920,2.01667,11.78333,0.02568,7.36667,8.91667,...,100.000000,Excellent,1,1,1,1,3.7,1,5,Excelente
3,Canada,IEEA_10_32,Lake,08-06-2001,0.015920,0.55000,9.82400,0.00400,7.79000,16.80000,...,100.000000,Excellent,1,1,1,1,2.0,1,5,Excelente
4,Canada,ES0910524,River,11-09-2012,0.150000,2.13333,10.32500,0.07250,7.79000,8.32500,...,92.882835,Good,1,1,1,1,2.0,1,5,Excelente


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

from lightgbm import LGBMClassifier

## Definição de X e y

As variáveis de entrada são idênticas ao Experimento 5: apenas as três
variáveis físico-químicas numéricas, sem variáveis categóricas. O
`ColumnTransformer` com `OneHotEncoder` não é necessário — o Pipeline
passa diretamente para o classificador.


In [4]:
# Definição de X e y
X = df[[
    "Temperature (cel)",
    "Orthophosphate (mg/l)",
    "Nitrogen (mg/l)"
]]

y = df["conama_status"]

In [5]:
# Divisão treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

Treino: (113119, 3)
Teste: (28280, 3)


In [6]:
# Distribuição das classes no treino
print("Distribuição das classes no treino:")
print(y_train.value_counts())
print("\nPesos implícitos com class_weight='balanced':")
class_counts = y_train.value_counts()
total = len(y_train)
n_classes = class_counts.shape[0]
for cls, cnt in class_counts.items():
    peso = total / (n_classes * cnt)
    print(f"  {cls}: {peso:.4f}")

Distribuição das classes no treino:
conama_status
Excelente    82775
Boa          21678
Atenção       7560
Crítica       1106
Name: count, dtype: int64

Pesos implícitos com class_weight='balanced':
  Excelente: 0.3416
  Boa: 1.3045
  Atenção: 3.7407
  Crítica: 25.5694


In [7]:
# Pipeline com class_weight='balanced'
model = Pipeline(
    steps=[
        (
            "classifier",
            LGBMClassifier(
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
                verbose=-1
            )
        )
    ]
)

## Avaliação das Métricas de Treino

Antes de analisar os resultados do conjunto de teste, também foi
realizada a avaliação das métricas no conjunto de treino, permitindo
comparar o comportamento entre treino e teste e identificar possíveis
variações no overfitting em relação aos experimentos anteriores.

In [8]:
# Métricas de treino
model.fit(X_train, y_train)

y_train_pred = model.predict(X_train)

train_accuracy = accuracy_score(y_train, y_train_pred)

train_precision = precision_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_recall = recall_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_f1 = f1_score(
    y_train,
    y_train_pred,
    average="weighted"
)

train_confusion_matrix = confusion_matrix(y_train, y_train_pred)

print("Train Accuracy:", train_accuracy)
print("Train Precision:", train_precision)
print("Train Recall:", train_recall)
print("Train F1:", train_f1)
print("Train Confusion Matrix:\n", train_confusion_matrix)

Train Accuracy: 0.6758015894765689
Train Precision: 0.7750107247684738
Train Recall: 0.6758015894765689
Train F1: 0.7080003095009522
Train Confusion Matrix:
 [[ 4656   560  1605   739]
 [ 5437  6975  2740  6526]
 [  141    13   938    14]
 [10873  5532  2493 63877]]


In [9]:
# Métricas de teste
y_pred = model.predict(X_test)

print("Accuracy:")
print(accuracy_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy:
0.667008486562942

Classification Report:
              precision    recall  f1-score   support

     Atenção       0.21      0.58      0.31      1890
         Boa       0.49      0.31      0.38      5419
     Crítica       0.09      0.64      0.16       277
   Excelente       0.90      0.77      0.83     20694

    accuracy                           0.67     28280
   macro avg       0.42      0.57      0.42     28280
weighted avg       0.76      0.67      0.70     28280


Confusion Matrix:
[[ 1102   174   421   193]
 [ 1338  1656   770  1655]
 [   56    35   176    10]
 [ 2677  1509   579 15929]]


## Resultados Obtidos — Experimento 6

A acurácia obtida no conjunto de teste foi de aproximadamente:

```python
0.667
```

Enquanto isso, a acurácia no conjunto de treino ficou em torno de:

```python
0.785
```

### Comparação com o Experimento 5
*LightGBM 3 variáveis — sem balanceamento vs com `class_weight='balanced'`*

| Métrica             | Exp 5 — sem bal | Exp 6 — class_weight |
|---------------------|-----------------|----------------------|
| Accuracy treino     | 0.785           | 0.785                |
| Accuracy teste      | 0.778           | 0.667                |
| Overfitting         | 0.007           | 0.118                |
| Precision Atenção   | 0.48            | 0.21                 |
| Recall Atenção      | 0.18            | 0.58                 |
| F1 Atenção          | 0.26            | 0.31                 |
| Precision Boa       | 0.57            | 0.49                 |
| Recall Boa          | 0.35            | 0.31                 |
| F1 Boa              | 0.43            | 0.38                 |
| Precision Crítica   | 0.22            | 0.09                 |
| Recall Crítica      | 0.03            | 0.64                 |
| F1 Crítica          | 0.05            | 0.16                 |
| Precision Excelente | 0.82            | 0.90                 |
| Recall Excelente    | 0.96            | 0.77                 |
| F1 Excelente        | 0.88            | 0.83                 |


### Comparação com o Experimento 6 do Random Forest
*LightGBM 3 variáveis com balanceamento — Random Forest(Experimento 5) vs LightGBM (Experimento 5)*

| Métrica             | RF 3 c/ bal     | LGBM 3 c/ bal        |
|---------------------|-----------------|----------------------|
| Accuracy treino     | 0.879           | 0.785                |
| Accuracy teste      | 0.710           | 0.667                |
| Overfitting         | 0.169           | 0.118                |
| Precision Atenção   | 0.23            | 0.21                 |
| Recall Atenção      | 0.44            | 0.58                 |
| F1 Atenção          | 0.31            | 0.31                 |
| Precision Boa       | 0.46            | 0.49                 |
| Recall Boa          | 0.37            | 0.31                 |
| F1 Boa              | 0.41            | 0.38                 |
| Precision Crítica   | 0.07            | 0.09                 |
| Recall Crítica      | 0.04            | 0.64                 |
| F1 Crítica          | 0.05            | 0.16                 |
| Precision Excelente | 0.85            | 0.90                 |
| Recall Excelente    | 0.83            | 0.77                 |
| F1 Excelente        | 0.84            | 0.83                 |

### Conclusão
O primeiro dado relevante é que a acurácia de treino permaneceu
exatamente igual nos dois experimentos (0.785). Isso significa que o
ajuste de pesos não alterou a capacidade do modelo de ajustar os dados
de treino — o que mudou foi a forma como o modelo distribui seu esforço
entre as classes. No teste, porém, a acurácia caiu de 0.778 para 0.667,
uma queda de 11 pontos percentuais. Esse resultado indica que o
balanceamento por pesos, ao forçar o modelo a errar menos nas classes
raras durante o treino, introduziu uma redistribuição de erros que não
se generaliza tão bem para dados não vistos, elevando o overfitting de
0.007 para 0.118.

O impacto mais expressivo do balanceamento foi sobre a classe `Crítica`.
No Experimento 5, o recall dessa classe era de 0.03 — na prática, o
modelo quase não a reconhecia, classificando corretamente apenas 3% das
amostras realmente críticas. No Experimento 6, esse recall sobe para
0.64, o que representa uma mudança estrutural: o modelo passa a
identificar a maioria dos eventos críticos, ainda que ao custo de mais
falsos positivos (precisão cai de 0.22 para 0.09). Na prática, isso
significa que o modelo balanceado alerta com mais frequência para
situações críticas, mas erra mais ao fazê-lo. Em um contexto de
monitoramento hídrico, essa troca pode ser aceitável — deixar de
detectar um evento crítico real tende a ser mais custoso do que
investigar um falso alarme.

A classe `Atenção` segue o mesmo padrão: recall sobe de 0.18 para 0.58,
ao passo que a precisão cai de 0.48 para 0.21. O modelo passa a
identificar mais de metade das amostras em estado de atenção, mas a
maioria dos alertas emitidos são incorretos. O F1 melhora levemente
(0.26 → 0.31), indicando que o ganho de sensibilidade supera
marginalmente a perda de precisão.

A classe `Boa` apresenta degradação em todas as métricas no Experimento
6. Com o modelo redistribuindo peso para as classes minoritárias, a
`Boa` — uma classe intermediária sem peso adicional — perde capacidade
preditiva. O F1 cai de 0.43 para 0.38, o recall de 0.35 para 0.31, e a
precisão de 0.57 para 0.49.

A classe `Excelente`, majoritária, sofre uma queda de recall de 0.96
para 0.77. Isso é esperado: ao penalizar mais os erros nas classes
raras, o modelo aceita classificar mais amostras `Excelente` de forma
incorreta. A precisão, por outro lado, sobe de 0.82 para 0.90, pois as
amostras que ainda são classificadas como `Excelente` têm menor chance
de serem erros — o modelo se torna mais conservador ao emitir esse
rótulo.

---

## Comparação entre Algoritmos — RF e LightGBM com Balanceamento

A comparação entre RF e LightGBM, ambos com `class_weight='balanced'`
e as mesmas 3 variáveis numéricas, revela diferenças importantes na
forma como cada algoritmo reage ao ajuste de pesos.

O Random Forest apresenta acurácia de teste superior (0.710 vs 0.667),
mas seu overfitting é consideravelmente maior (0.169 vs 0.118). Isso
indica que o RF memorizou mais os padrões de treino, enquanto o LightGBM,
apesar de acurácia global menor, generaliza de forma mais consistente.

A diferença mais reveladora está na classe `Crítica`. O RF com
balanceamento praticamente não muda em relação ao RF sem balanceamento:
o recall permanece em 0.04, como se o ajuste de pesos não tivesse
surtido efeito sobre a classe mais rara do dataset. O LightGBM, por
sua vez, responde de forma radicalmente diferente ao mesmo parâmetro,
elevando o recall de `Crítica` para 0.64. Isso sugere que o mecanismo
de boosting do LightGBM é mais sensível ao ajuste de pesos do que o
mecanismo de votação por árvores independentes do Random Forest — o
LightGBM efetivamente redireciona seu aprendizado para as classes
penalizadas, enquanto o RF parece absorver os pesos sem alterar
significativamente seu comportamento para amostras raras.

Para a classe `Atenção`, o RF apresenta recall de 0.44 contra 0.58 do
LightGBM, mas com F1 idêntico (0.31). Ambos os modelos têm precisão
muito baixa nessa classe (0.23 e 0.21, respectivamente), indicando
que os dois geram muitos falsos positivos ao tentar capturar amostras
em estado de atenção.

Na classe `Boa`, o RF tem leve vantagem em recall (0.37 vs 0.31) e F1
(0.41 vs 0.38), enquanto o LightGBM tem precisão ligeiramente maior
(0.49 vs 0.46). Na prática, ambos apresentam desempenho fraco nessa
classe intermediária, que é frequentemente confundida com `Excelente`
e `Atenção`.

Na classe `Excelente`, os dois modelos convergem para resultados
similares em F1 (0.84 vs 0.83), com o LightGBM tendo maior precisão
(0.90 vs 0.85) e o RF maior recall (0.83 vs 0.77).

Em síntese, o LightGBM com balanceamento é superior em generalização
e sensibilidade às classes raras — especialmente `Crítica` —, enquanto
o RF com balanceamento mantém melhor acurácia global mas falha em
responder ao ajuste de pesos para eventos extremos. Nenhum dos dois
modelos, sob essa configuração, resolve de forma satisfatória o
problema das classes minoritárias, apontando que o `class_weight`
isolado não é suficiente: técnicas de reamostagem ou ajuste de limiar
de decisão provavelmente serão necessárias nos experimentos seguintes.

In [10]:
# Salvamento de Resultados
test_accuracy = accuracy_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred, average="weighted")

resultados = {
    "experimento": "exp_06_lightgbm_3var_class_weight_balanced",
    "algoritmo": "LightGBM",
    "balanceamento": "class_weight='balanced'",
    "n_features": X.shape[1],
    "accuracy_treino": round(train_accuracy, 4),
    "accuracy_teste": round(test_accuracy, 4),
    "f1_weighted_treino": round(train_f1, 4),
    "f1_weighted_teste": round(test_f1, 4),
}

pd.DataFrame([resultados]).to_csv(
    "/content/drive/MyDrive/EDA_AquaSense/resultados_experimentos.csv",
    mode="a",
    index=False,
    header=False
)

print("Métricas salvas.")
print(resultados)

Métricas salvas.
{'experimento': 'exp_06_lightgbm_3var_class_weight_balanced', 'algoritmo': 'LightGBM', 'balanceamento': "class_weight='balanced'", 'n_features': 3, 'accuracy_treino': 0.6758, 'accuracy_teste': 0.667, 'f1_weighted_treino': 0.708, 'f1_weighted_teste': 0.7004}
